# **Implementation of a single-item EOQ model**

In [44]:
# import the nonlinear programming solver
import sys, os
import numpy as np
import pandas as pd
# if 'google.colab' in sys.modules:
# %pip install idaes-pse --pre >/dev/null 2>/dev/null
# !idaes get-extensions --to ./bin
# os.environ['PATH'] += ':bin'
solver = "ipopt"

import pyomo.kernel as pmo
SOLVER = pmo.SolverFactory(solver)
assert SOLVER.available(), f"Solver {solver} is not available."



# **Pyomo modeling with conic.quadratic**

In [45]:
h = 0.75  # cost of holding one time for one year
c = 500.0  # cost of processing one order
d = 10000.0  # annual demand

m = pmo.block()

# define variables for conic constraints
m.x = pmo.variable(lb=0)
m.y = pmo.variable(lb=0)

# second-order cone constraint x[0]^2 + … + x[n-1]^2 <= r^2,
# pmo.conic.quadratic(r, x)
#.as_domain: Builds a conic domain. Input arguments take the same form as those of the conic constraint,
# but in place of each variable, one can optionally supply a constant, linear expression, or None.
m.q = pmo.conic.quadratic.as_domain(m.x+m.y, [2, m.x-m.y])

# linear objective
m.eoq = pmo.objective(h * m.x / (2*d) + c *  m.y)

# solve
SOLVER.solve(m)

# solution
print(f"\nEOQ = {m.x():0.2f}")


EOQ = 3651.48


# **Pyomo Modeling with conic.rotated_quadratic**

Alternatively, we can directly use pmo.conic.rotated_quadratic.as_domain(r1, r2, x) for a rotated quadratic conic constraint of the form:
x[0]^2 + … + x[n-1]^2 <= 2*r1*r2.
** Pay attention to the coefficient 2 on the right side of the above standard form. For our constraint xy >= 1, we need to write it as 2xy >= 2, and let r1 =x, r2 = y and x=sqrt{2} in the function.

Also note the use of .as_domain() eliminates the need to specify a variable z = 2

In [46]:
# rotated secnd-order conic constraint
m.q = pmo.conic.rotated_quadratic.as_domain(m.x, m.y, [np.sqrt(2)])

# linear objective
m.eoq = pmo.objective(h * m.x / (2*d) + c *  m.y)

# solve
SOLVER.solve(m)

# solution
print(f"\nEOQ = {m.x():0.2f}")

object (type=block). This is usually indicative of a modeling error. To avoid
this warning, delete the original object from the block before assigning a new
object.
object (type=objective). This is usually indicative of a modeling error. To
avoid this warning, delete the original object from the block before assigning
a new object.

EOQ = 3651.48


# **Data for Q3:**

In [47]:
import numpy as np
import pandas as pd

n = 30
B= 100000
df = pd.DataFrame()
df["h"] = np.random.uniform(0.5, 2.0, n)
df["c"] = np.random.randint(300, 500, n)
df["d"] = np.random.randint(100, 5000, n)
df["b"] = np.random.uniform(10, 50,n)
df.set_index(pd.Series(f"product {i:03d}" for i in range(n)), inplace=True)

# display(df)

In [48]:
model = pmo.block()

model.x = pmo.variable_list()
model.y = pmo.variable_list()

for i in range(n):
    model.x.append(pmo.variable(lb=0))
    model.y.append(pmo.variable(lb=0))
    
model.socp = pmo.block_list()

for i in range(n):
    model.socp.append(pmo.conic.quadratic.as_domain(model.x[i]+model.y[i], [2, model.x[i]-model.y[i]]))
    
model.place = pmo.constraint(expr=sum(df["b"].iloc[i] * model.x[i] for i in range(n)) <= B)



# linear objective
model.eoq = pmo.objective(sum(df["h"].iloc[i] * model.x[i] / (2*df["d"].iloc[i]) + df["c"].iloc[i] *  model.y[i] for i in range(n)), 
                          sense=pmo.minimize)


In [49]:
SOLVER.solve(model)

df["EOQ"] = [model.x[i]() for i in range(n)]

In [50]:
df

,h,c,d,b,EOQ
product 000,0.986184,408,373,45.267395,96.579461
product 001,1.580929,362,3504,31.536170,110.257933
product 002,0.941114,364,1634,14.890290,159.872769
product 003,1.439814,365,223,30.734114,106.752275
product 004,0.525807,359,658,12.358906,173.102538
product 005,1.650510,339,2737,26.119889,116.967962
product 006,0.959906,369,3305,22.692554,131.281934
product 007,1.776920,419,830,30.123070,119.587709
product 008,1.474959,382,1715,16.377068,155.605208
product 009,1.048687,327,4662,23.536767,121.452744
